In [9]:
!unzip aggregated.zip

Archive:  aggregated.zip
   creating: sp26cai6108mle-project-aggregated/
   creating: sp26cai6108mle-project-aggregated/aggregated/
   creating: sp26cai6108mle-project-aggregated/aggregated/backpack/
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img1.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img100.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img102.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img103.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img104.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img105.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img106.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img108.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img109.png  
  inflating: sp26cai6108mle-project-aggregated/aggregated/backpack/img110.png  
  

In [10]:
!mv sp26cai6108mle-project-aggregated mle-aggregated

In [11]:
LABEL_ORDER = [
    "pen", "paper", "book", "clock", "phone", "laptop",
    "chair", "desk", "bottle", "keychain", "backpack", "calculator"
]

In [12]:
import torch

LABEL_TO_IDX = {label: i for i, label in enumerate(LABEL_ORDER)}

def folder_to_multihot(folder_name):
    labels = folder_name.split("_")
    target = torch.zeros(len(LABEL_ORDER), dtype=torch.float32)

    for label in labels:
        target[LABEL_TO_IDX[label]] = 1.0

    return target

In [13]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [14]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset

LABEL_ORDER = [
    "pen", "paper", "book", "clock", "phone", "laptop",
    "chair", "desk", "bottle", "keychain", "backpack", "calculator"
]
LABEL_TO_IDX = {label: i for i, label in enumerate(LABEL_ORDER)}
VALID_LABELS = set(LABEL_ORDER)

class MultiLabelFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        for folder in self.root_dir.iterdir():
            if not folder.is_dir():
                continue

            labels = folder.name.split("_")

            # skip invalid folders
            if any(label not in VALID_LABELS for label in labels):
                continue
            if len(labels) != len(set(labels)):
                continue

            target = torch.zeros(len(LABEL_ORDER), dtype=torch.float32)
            for label in labels:
                target[LABEL_TO_IDX[label]] = 1.0

            for img_path in folder.glob("*.png"):
                self.samples.append((img_path, target.clone()))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, target = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, target

In [15]:
from torch.utils.data import DataLoader, Subset
import torch

data_root = "/content/mle-aggregated/aggregated"

full_dataset = MultiLabelFolderDataset(data_root, transform=None)

n = len(full_dataset)
train_size = int(0.70 * n)
val_size = int(0.15 * n)
test_size = n - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_subset, val_subset, test_subset = torch.utils.data.random_split(
    range(n),
    [train_size, val_size, test_size],
    generator=generator
)

train_indices = train_subset.indices
val_indices = val_subset.indices
test_indices = test_subset.indices

train_dataset = Subset(
    MultiLabelFolderDataset(data_root, transform=train_transforms),
    train_indices
)

val_dataset = Subset(
    MultiLabelFolderDataset(data_root, transform=val_transforms),
    val_indices
)

test_dataset = Subset(
    MultiLabelFolderDataset(data_root, transform=val_transforms),
    test_indices
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 3180
Val size: 681
Test size: 682


In [16]:
img, target = train_dataset[0]
print(img.shape)      # should look like [3, 128, 128]
print(target.shape)   # should look like [12]
print(target)

torch.Size([3, 224, 224])
torch.Size([12])
tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1.])


**Train Model**

In [17]:
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=30,
    threshold=0.4,
    save_path="best_model.pth",
    force_one_prediction=True,
    patience=5
):
    model = model.to(device)

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_epoch = -1
    epochs_without_improvement = 0

    final_all_labels = None
    final_all_preds = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).float()

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).float()

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                probs = torch.sigmoid(outputs)
                preds = (probs > threshold).int()

                if force_one_prediction:
                    top_idx = torch.argmax(probs, dim=1)
                    empty_rows = preds.sum(dim=1) == 0
                    preds[empty_rows, top_idx[empty_rows]] = 1

                all_labels.append(labels.cpu().int())
                all_preds.append(preds.cpu())

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        all_labels = torch.cat(all_labels).numpy()
        all_preds = torch.cat(all_preds).numpy()

        final_all_labels = all_labels
        final_all_preds = all_preds

        micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
        macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model at epoch {best_epoch}")
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch+1}, "
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}, "
            f"Micro F1: {micro_f1:.4f}, "
            f"Macro F1: {macro_f1:.4f}"
        )

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after epoch {epoch+1}")
            break

    print(f"Best epoch: {best_epoch}, Best Val Loss: {best_val_loss:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss")
    plt.plot(range(1, len(val_losses) + 1), val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.show()

    return {
        "model": model,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "all_labels": final_all_labels,
        "all_preds": final_all_preds
    }

**Simple CNN**

In [21]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
simplecnn_model = SimpleCNN(num_classes=12).to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(simplecnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

cnn_results = train_model(
    model=simplecnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=30,
    threshold=0.4,
    save_path="best_simplecnn.pth"
)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

resnet_model = models.resnet18(pretrained=True)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 12)
resnet_model = resnet_model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(resnet_model.parameters(), lr=1e-4, weight_decay=1e-4)

resnet_results = train_model(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=30,
    threshold=0.4,
    save_path="best_resnet.pth"
)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score

def find_best_thresholds(model, loader, device, num_classes=12):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).float()

            logits = model(images)
            probs = torch.sigmoid(logits)

            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    thresholds = []
    best_class_f1s = []

    for c in range(num_classes):
        best_t = 0.5
        best_f1 = -1.0

        for t in np.arange(0.1, 0.9, 0.05):
            preds = (all_probs[:, c] >= t).astype(int)
            f1 = f1_score(all_labels[:, c], preds, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)

        thresholds.append(best_t)
        best_class_f1s.append(best_f1)

    return thresholds, best_class_f1s

In [ ]:
model = resnet_model
all_labels = resnet_results["all_labels"]
all_preds = resnet_results["all_preds"]

In [ ]:
best_thresholds, tuned_class_f1s = find_best_thresholds(
    model,
    val_loader,
    device,
    num_classes=len(LABEL_ORDER)
)

print("Best thresholds:", best_thresholds)
print("Best per-class F1 during search:", tuned_class_f1s)

In [ ]:
torch.save(model.state_dict(), "resnet_mle.pth")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

per_class_precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
per_class_recall = recall_score(all_labels, all_preds, average=None, zero_division=0)
per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

for i, name in enumerate(LABEL_ORDER):
    print(
        f"{name:12s} | "
        f"Precision: {per_class_precision[i]:.4f} | "
        f"Recall: {per_class_recall[i]:.4f} | "
        f"F1: {per_class_f1[i]:.4f}"
    )

**Evaluate with Thresholds**

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import torch

def evaluate_with_thresholds(model, data_loader, best_thresholds, device, label_order):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    thresholds_tensor = torch.tensor(best_thresholds, dtype=torch.float32).view(1, -1).to(device)

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device).float()

            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs >= thresholds_tensor).int()

            all_labels.append(labels.cpu().int())
            all_preds.append(preds.cpu())
            all_probs.append(probs.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_preds = torch.cat(all_preds).numpy()
    all_probs = torch.cat(all_probs).numpy()

    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    per_class_precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    per_class_recall = recall_score(all_labels, all_preds, average=None, zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return {
        "all_labels": all_labels,
        "all_preds": all_preds,
        "all_probs": all_probs,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "per_class_precision": per_class_precision,
        "per_class_recall": per_class_recall,
        "per_class_f1": per_class_f1,
        "label_order": label_order,
        "best_thresholds": best_thresholds
    }

In [ ]:
def print_evaluation_results(results, split_name="Validation", show_thresholds=True):
    print(f"{split_name} Micro F1: {results['micro_f1']:.4f}")
    print(f"{split_name} Macro F1: {results['macro_f1']:.4f}")

    for i, name in enumerate(results["label_order"]):
        if show_thresholds:
            print(
                f"{name:12s} | "
                f"Threshold: {results['best_thresholds'][i]:.2f} | "
                f"Precision: {results['per_class_precision'][i]:.4f} | "
                f"Recall: {results['per_class_recall'][i]:.4f} | "
                f"F1: {results['per_class_f1'][i]:.4f}"
            )
        else:
            print(
                f"{name:12s} | "
                f"Precision: {results['per_class_precision'][i]:.4f} | "
                f"Recall: {results['per_class_recall'][i]:.4f} | "
                f"F1: {results['per_class_f1'][i]:.4f}"
            )

In [ ]:
val_results = evaluate_with_thresholds(
    model=model,
    data_loader=val_loader,
    best_thresholds=best_thresholds,
    device=device,
    label_order=LABEL_ORDER
)

print_evaluation_results(val_results, split_name="Validation", show_thresholds=True)

In [ ]:
test_results = evaluate_with_thresholds(
    model=model,
    data_loader=test_loader,
    best_thresholds=best_thresholds,
    device=device,
    label_order=LABEL_ORDER
)

print_evaluation_results(test_results, split_name="Test", show_thresholds=False)

**Shared Prediction Logic**

In [ ]:
def get_probabilities(batch_images):
    model.eval()
    with torch.no_grad():
        logits = model(batch_images.to(device))
        probs = torch.sigmoid(logits).cpu()
    return probs

**Grad CAM**

In [ ]:
!pip -q install grad-cam


In [ ]:
# Citation: https://arxiv.org/pdf/1610.02391
# https://github.com/jacobgil/pytorch-grad-cam GradCam Github REPO

import numpy as np
import matplotlib.pyplot as plt
import torch

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ResNet target layer
target_layers = [model.layer4[-1]]

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def unnormalize_image(img_tensor):
    img = img_tensor.detach().cpu() * std + mean
    img = img.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)

def get_batch_predictions(batch_images, threshold=0.4, thresholds=None):
    probs = get_probabilities(batch_images)

    if thresholds is not None:
        thresholds_tensor = torch.tensor(thresholds, dtype=torch.float32).view(1, -1)
        preds = (probs >= thresholds_tensor).int()
    else:
        preds = (probs >= threshold).int()

    return probs, preds

def labels_from_multihot(multihot, class_names=LABEL_ORDER):
    return [class_names[i] for i, v in enumerate(multihot) if int(v) == 1]

def show_gradcam_for_class(image_tensor, class_idx, alpha=0.5):
    cam = GradCAM(model=model, target_layers=target_layers)
    input_tensor = image_tensor.unsqueeze(0).to(device)
    targets = [ClassifierOutputTarget(class_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    rgb_img = unnormalize_image(image_tensor)
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True, image_weight=alpha)
    return rgb_img, visualization

def show_example_with_gradcam(batch_images, batch_labels, image_index=0, class_idx=None, threshold=0.4, thresholds=None):
    probs, preds = get_batch_predictions(batch_images, threshold=threshold, thresholds=thresholds)

    image_tensor = batch_images[image_index].cpu()
    true_labels = labels_from_multihot(batch_labels[image_index].cpu())
    pred_labels = labels_from_multihot(preds[image_index])

    if class_idx is None:
        class_idx = int(torch.argmax(probs[image_index]).item())

    rgb_img, visualization = show_gradcam_for_class(image_tensor, class_idx)

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(rgb_img)
    plt.axis('off')
    plt.title(f'Original\nTrue: {true_labels}\nPred: {pred_labels}')

    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.axis('off')
    plt.title(f'Grad-CAM for {LABEL_ORDER[class_idx]}\nProb: {probs[image_index][class_idx]:.3f}')

    plt.tight_layout()
    plt.show()

    return probs[image_index], preds[image_index]

def show_all_predicted_gradcams(batch_images, batch_labels, image_index=0, threshold=0.4, thresholds=None):
    probs, preds = get_batch_predictions(batch_images, threshold=threshold, thresholds=thresholds)
    image_tensor = batch_images[image_index].cpu()
    true_labels = labels_from_multihot(batch_labels[image_index].cpu())
    pred_class_indices = [i for i, v in enumerate(preds[image_index]) if int(v) == 1]
    rgb_img = unnormalize_image(image_tensor)

    if len(pred_class_indices) == 0:
        print('No predicted labels above threshold for this image.')
        plt.figure(figsize=(4, 4))
        plt.imshow(rgb_img)
        plt.axis('off')
        plt.title(f'True: {true_labels}')
        plt.show()
        return

    fig, axes = plt.subplots(1, len(pred_class_indices) + 1, figsize=(5 * (len(pred_class_indices) + 1), 5))
    axes[0].imshow(rgb_img)
    axes[0].axis('off')
    axes[0].set_title(f'Original\nTrue: {true_labels}')

    for ax, class_idx in zip(axes[1:], pred_class_indices):
        _, visualization = show_gradcam_for_class(image_tensor, class_idx)
        ax.imshow(visualization)
        ax.axis('off')
        ax.set_title(f'{LABEL_ORDER[class_idx]}\nProb: {probs[image_index][class_idx]:.3f}')

    plt.tight_layout()
    plt.show()


In [ ]:
batch_images, batch_labels = next(iter(val_loader))

show_all_predicted_gradcams(
    batch_images,
    batch_labels,
    image_index=5,
    thresholds=best_thresholds if "best_thresholds" in globals() else None
)

**LIME**

In [ ]:
!pip install lime

In [ ]:
import numpy as np
import torch

def predict_fn(images_np, model, device):
    model.eval()

    # LIME passes images as (N, H, W, C)
    images_np = np.array(images_np, dtype=np.float32)

    # scale to 0-1
    if images_np.max() > 1.0:
        images_np = images_np / 255.0

    # Convert to tensor and change to (N, C, H, W)
    images_tensor = torch.tensor(images_np, dtype=torch.float32).permute(0, 3, 1, 2)

    # Normalize the same way inputs were normalized
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)
    images_tensor = (images_tensor - mean) / std

    images_tensor = images_tensor.to(device)

    with torch.no_grad():
        logits = model(images_tensor)
        probs = torch.sigmoid(logits)

    return probs.cpu().numpy()

In [ ]:
# Citation:
# https://arxiv.org/abs/1602.04938
# https://github.com/marcotcr/lime

from lime import lime_image

def explain_with_lime(image_np, model, device, class_names, target_class_idx):
    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        image=image_np,
        classifier_fn=lambda x: predict_fn(x, model, device),
        top_labels=None,
        labels=(target_class_idx,),
        hide_color=0,
        num_samples=1000
    )

    temp, mask = explanation.get_image_and_mask(
        label=target_class_idx,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    return temp, mask

In [ ]:
# Citation:
# https://arxiv.org/abs/1602.04938
# https://github.com/marcotcr/lime

import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries

def show_lime_explanation(image_np, temp, mask, class_name):
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(image_np)
    plt.axis("off")
    plt.title("Original Image")

    plt.subplot(1, 2, 2)
    plt.imshow(mark_boundaries(temp, mask))
    plt.axis("off")
    plt.title(f"LIME for {class_name}")

    plt.tight_layout()
    plt.show()

In [ ]:
probs, preds = get_batch_predictions(batch_images)
img = 4
class_idx = int(torch.argmax(probs[img]).item())

image_tensor = batch_images[img].cpu()
image_np = unnormalize_image(image_tensor)

temp, mask = explain_with_lime(
    image_np=image_np,
    model=model,
    device=device,
    class_names=LABEL_ORDER,
    target_class_idx=class_idx
)

In [ ]:
show_lime_explanation(image_np, temp, mask, LABEL_ORDER[class_idx])